In [1]:
from sequana import DNA
from sequana import FastA
import pandas as pd
import numpy as np
from sklearn.preprocessing import  StandardScaler
import re
from scipy.signal import find_peaks
import matplotlib.pyplot as plt
from scipy.ndimage import uniform_filter1d
from scipy.signal import savgol_filter
from scipy.signal import argrelextrema
from itertools import combinations

from scipy.signal import peak_widths


In [2]:
def count_homopolymers(seq, min_length=5):
    # Ex : trouve AAAAA ou TTTTT, etc.
    pattern = re.compile(rf"(A{{{min_length},}}|T{{{min_length},}}|C{{{min_length},}}|G{{{min_length},}})")
    return len(pattern.findall(seq.upper()))

def load_fasta(fasta_path, window_size=100):
    f = FastA(fasta_path)
    data = []


    for maseq in f:

        print(maseq.name)
        
        features = []

        s = DNA(maseq.sequence.upper())
        seq = maseq.sequence.upper()
        s.window = window_size

       

        #Homopolymere
        X2 = []
        X3 = []

        
        for i in range(0, len(seq)-2, 1):
            
            window = seq[max(0, i - window_size//2):min(i+window_size//2,len(seq))]
            nb = count_homopolymers(window, min_length=2)
            X2.append(nb)

            nb = count_homopolymers(window, min_length=3)
            X3.append(nb)
        


            
        X2 = X2[5000:-5000]
        X3 = X3[5000:-5000]


 






   

        df= pd.DataFrame({
            'X2':X2,
            'X3':X3
        })

        
        data.append(df)

            
    return data

 
data = load_fasta("../data/Fasta/TriTrypDB-68_LpanamensisMHOMPA94PSC1_Genome.fasta",200)


CP009370
CP009371
CP009372
CP009373
CP009374
CP009375
CP009376
CP009377
CP009378
CP009379
CP009380
CP009381
CP009382
CP009383
CP009384
CP009385
CP009386
CP009387
CP009388
CP009389
CP009390
CP009391
CP009392
CP009393
CP009394
CP009395
CP009396
CP009397
CP009398
CP009399
CP009400
CP009401
CP009402
CP009403
CP009404


In [25]:
df = pd.read_csv("../data/Centromere_Positions/pos_libre.csv")
pos_libre = {
    int(row.Chromosome): (int(row.Start), int(row.End))
    for row in df.itertuples(index=False)
}

vecteur_Debut = []
vecteur_Fin = []

vecteur_longeur = []


g = 0


for i in range(0,36):
    temp_ =  data[i]['X3']*data[i]['X2']
    temp_ = temp_[40000:-2000]

    mean = np.mean(temp_)

    peaks, properties = find_peaks(temp_,  prominence=mean*2, distance=20)

    window = 3000
    peak_medians = []

 



        #Moyenne
    peak_means = []
    for peak in peaks:
        start = max(0, peak - window//2)
        end = min(len(temp_), peak + window//2)
        mean_val = np.mean(temp_[start:end])
        peak_medians.append((peak, mean_val))
    
  #  for peak in peaks:
  #      start = max(0, peak - window//2)
  #      end = min(len(temp_), peak + window//2 + 1)
    
   #     neighborhood = temp_[start:end]
   #     if len(neighborhood) > 0:
   #         median_val = np.median(neighborhood)
   #         peak_medians.append((peak, median_val))
    
    # Trouver le pic avec la médiane la plus élevée
    if peak_medians:
        peak_max, max_median = max(peak_medians, key=lambda x: x[1])
        max_index = peak_max


        #Chercher la taille du centromere


        # Appliquer un filtre pour lisser
        temp_X2_3 = uniform_filter1d(temp_, size=1500)
        temp_X2_3 = temp_X2_3[max_index-25000:max_index+25000]
        

        
        # Détection des pics
        peaks, properties = find_peaks(temp_X2_3, prominence=5, distance=200)

            
        # Trouver le pic avec la plus grande **prominence**
        if len(peaks) > 0:
            prominences = properties['prominences']
            peak_index = np.argmax(prominences)
            peak_max = peaks[peak_index]
        
            # Calcule la largeur du pic à mi-hauteur
            widths_result = peak_widths(temp_X2_3, peaks, rel_height=0.4)
        
            taille = widths_result[0][peak_index]
            debut = widths_result[2][peak_index]
            fin = widths_result[3][peak_index]
            seuil =  widths_result[1][peak_index]


      
            debut = int(debut)
            fin = int(fin)
            
            marge = 750
            finish = True
            distance = []
            temp_fin = []
            while finish:
                
                recherche = False
                limiteFin = min(len(temp_X2_3), fin + marge)
                pos = fin

                while pos < limiteFin and recherche == False:
                    if  temp_X2_3[pos] > seuil:
                        recherche = True
                    pos += 1
                if recherche == True:
                     temp_fin.append(fin)
                     distance.append(0)
                     while pos + 1 < len(temp_X2_3) and temp_X2_3[pos] > seuil:
                        pos += 1
                        distance[len(distance)-1] += 1

                     fin = pos
                else: 
                    finish = False

            t = len(distance)-1
            while t >= 0 and distance[t] < 200:
                fin = temp_fin[t]
                t = t -1 
            #Avant
            finish = True
            distance = 0
            distance = []
            temp_debut = []

            while finish:
                recherche = False
                limiteDebut = max(0, debut - marge)
                pos = debut
                while pos > limiteDebut and recherche == False:
 
                    if  temp_X2_3[pos] > seuil:
                        recherche = True
                    pos -= 1
        
                if recherche == True:
                     temp_debut.append(debut)
                     distance.append(0)

                     while pos + 1 < len(temp_X2_3) and temp_X2_3[pos] > seuil:
                        pos -= 1
                        distance[len(distance)-1] += 1
                     debut = pos
                else:
                    finish = False


            t = len(distance)-1
            while t >= 0 and distance[t] < 200:
                debut = temp_debut[t]
                t = t -1 
                

            
            
    
            debut = debut+max_index-25000+5000+40000
            fin = fin+max_index-25000+5000+40000
            debut = int(debut)
            fin = int(fin)
            max_index = max_index+5000+40000
            taille = fin - debut
            # Affichage

        
    
            pos_start, pos_end = pos_libre[i+1]

            if abs(pos_start-debut) > 50000:
                print(f'{i+1} : Commence {debut}    Fini {fin}  Taille : {taille} Attention !!! Diff position : {pos_start-debut}')
            else:
                print(f'{i+1} : Commence {debut}    Fini {fin}  Taille : {taille}')

   

   



            vecteur_Debut.append(debut)
            vecteur_Fin.append(fin)
            vecteur_longeur.append(taille)


print(len(vecteur_Debut))
result = pd.DataFrame()
result['Chromosome'] = list(range(1, 37))
result['start'] = vecteur_Debut 
result['end'] = vecteur_Fin
result['length'] = vecteur_longeur
result.to_csv(f"../output/estimation/panamensis.csv", index=False)

1 : Commence 240613    Fini 242026  Taille : 1413
2 : Commence 220598    Fini 223837  Taille : 3239
3 : Commence 246228    Fini 249803  Taille : 3575
4 : Commence 121979    Fini 125728  Taille : 3749
5 : Commence 367716    Fini 372051  Taille : 4335
6 : Commence 118156    Fini 121569  Taille : 3413
7 : Commence 205003    Fini 208581  Taille : 3578
8 : Commence 374750    Fini 377003  Taille : 2253 Attention !!! Diff position : 107307
9 : Commence 257382    Fini 260096  Taille : 2714
10 : Commence 249396    Fini 255633  Taille : 6237
11 : Commence 448421    Fini 450584  Taille : 2163 Attention !!! Diff position : -291097
12 : Commence 263020    Fini 266648  Taille : 3628
13 : Commence 128269    Fini 132495  Taille : 4226
14 : Commence 153274    Fini 160518  Taille : 7244
15 : Commence 324541    Fini 331494  Taille : 6953
16 : Commence 321420    Fini 326225  Taille : 4805
17 : Commence 285664    Fini 290882  Taille : 5218 Attention !!! Diff position : 52630
18 : Commence 427162    Fini 42

In [7]:
for i in range(0,35):
    print(len(data[i]))

258609
253711
377597
421106
439153
501624
541244
426871
551582
502282
576831
481199
608780
574525
596996
671772
613272
668716
631447
2409224
734058
634208
736971
822419
876797
1014525
1033454
1142634
1157639
1299974
1330776
1517380
1404951
1926230
2600167


In [10]:
chrom20 = data[19][1690410:]
chrom34 = data[19][:1690410]

data[19] = chrom20
data.append(data[34])
data[34] = data[33]
data[33] = chrom34

